# 02_VDDC
## Variable Definition and Data Check

This notebook documents the variable definitions, missing/invalid value handling, recoding rule, and final sample size used in the analysis.

Chosen variables:
- **Group variable:** `WhatIsYourSex`
- **Response variable:** `CurrentAlcoholUse`

## 1. Variable Definition

### Group variable: `WhatIsYourSex`
- **What it measures:** The biological sex of the student.
- **Valid codes used:** `1`, `2`
- **Meaning of code 1:** Female
- **Meaning of code 2:** Male
- **Recoding rule:** Rows with missing or invalid codes are dropped.

### Response variable: `CurrentAlcoholUse`
- **What it measures:** How often the student currently drinks alcohol.
- **Valid codes used:** `1`, `2`, `3`, `4`, `5`, `6`, `7`
- **Meaning of success (current user):** codes `2–7` → recoded as `1`
- **Meaning of failure (non-user):** code `1` → recoded as `0`
- **Recoding rule:** Rows with missing or invalid codes are dropped.

### Missing or invalid values
In this notebook, missing or invalid values are handled using **listwise deletion (dropna)**:
- Any row with a missing or invalid value in either variable is removed entirely.
- This approach is appropriate because both variables are categorical, and imputation of categorical group variables (especially sex) is not appropriate.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

input_path = Path("../data/processed/cycle3_selected_variables.csv")
output_dir1 = Path("../outputs/tables")
output_dir2 = Path("../data/processed")
output_dir1.mkdir(parents=True, exist_ok=True)
output_dir2.mkdir(parents=True, exist_ok=True)

missing_summary_path = output_dir1 / "missing_summary.csv"
output_path = output_dir2 / "cycle3_vddc_cleaned.csv"

## Load data

In [ ]:
df = pd.read_csv(input_path)

print("Original data shape:", df.shape)
display(df.describe().T)

## Step 1: Identify missing / invalid values

In [ ]:
group_col = "WhatIsYourSex"
response_col = "CurrentAlcoholUse"

df_check = df.copy()

# Group variable: only 1 and 2 are valid
valid_group_codes = [1, 2]
group_numeric = pd.to_numeric(df_check[group_col], errors="coerce")
group_missing_mask = group_numeric.isna()
group_invalid_mask = ~group_numeric.isin(valid_group_codes) & ~group_missing_mask
group_problem_mask = group_missing_mask | group_invalid_mask
group_valid_mask = ~group_problem_mask

# Response variable: only 1–7 are valid
valid_response_codes = [1, 2, 3, 4, 5, 6, 7]
response_numeric = pd.to_numeric(df_check[response_col], errors="coerce")
response_missing_mask = response_numeric.isna()
response_invalid_mask = ~response_numeric.isin(valid_response_codes) & ~response_missing_mask
response_problem_mask = response_missing_mask | response_invalid_mask
response_valid_mask = ~response_problem_mask

missing_summary = pd.DataFrame({
    "variable": [group_col, response_col],
    "what_it_measures": [
        "Biological sex of the student",
        "How often the student currently drinks alcohol"
    ],
    "valid_codes_or_values": [
        "1 (Female), 2 (Male)",
        "1–7"
    ],
    "missing_count": [
        int(group_missing_mask.sum()),
        int(response_missing_mask.sum())
    ],
    "invalid_count": [
        int(group_invalid_mask.sum()),
        int(response_invalid_mask.sum())
    ],
    "total_problem_count": [
        int(group_problem_mask.sum()),
        int(response_problem_mask.sum())
    ],
    "valid_count": [
        int(group_valid_mask.sum()),
        int(response_valid_mask.sum())
    ]
})

print("Missing / invalid value summary:")
display(missing_summary)

## Step 2: Drop missing / invalid rows

In [ ]:
df_clean = df.copy()

# Force numeric
df_clean[group_col] = pd.to_numeric(df_clean[group_col], errors="coerce")
df_clean[response_col] = pd.to_numeric(df_clean[response_col], errors="coerce")

# Drop rows where either variable is missing or invalid
df_clean = df_clean[
    df_clean[group_col].isin(valid_group_codes) &
    df_clean[response_col].isin(valid_response_codes)
].copy()

# Recode CurrentAlcoholUse: 1 = non-user (failure), 2–7 = current user (success)
df_clean["CurrentAlcoholUse_binary"] = df_clean[response_col].apply(
    lambda x: 1 if x in [2, 3, 4, 5, 6, 7] else 0
)

print(f"Rows after cleaning: {len(df_clean)}")
print(f"Rows removed: {len(df) - len(df_clean)}")
print()
display(df_clean.head())

## Step 3: Final sample size by group

In [ ]:
group_labels = {1: "Female", 2: "Male"}

final_info = (
    df_clean.groupby(group_col)["CurrentAlcoholUse_binary"]
    .agg(
        n="count",
        current_user_count="sum"
    )
    .reset_index()
)
final_info["sex_label"] = final_info[group_col].map(group_labels)
final_info["proportion_current_user"] = (
    final_info["current_user_count"] / final_info["n"]
).round(4)

print("Final sample size by group:")
display(final_info[[group_col, "sex_label", "n", "current_user_count", "proportion_current_user"]])

## Step 4: Save outputs

In [ ]:
missing_summary.to_csv(missing_summary_path, index=False)
df_clean.to_csv(output_path, index=False)

print(f"Saved missing summary to: {missing_summary_path}")
print(f"Saved processed data to: {output_path}")

## 2. Short Summary

This notebook clearly shows:
- variable name and what each variable measures
- valid codes used for each variable
- how missing or invalid values are handled using listwise deletion
- recoding rule for `CurrentAlcoholUse` (binary: 0 = non-user, 1 = current user)
- final sample size by group (Female vs Male)

Saved outputs:
- `../outputs/tables/missing_summary.csv`
- `../data/processed/cycle3_vddc_cleaned.csv`